# SmartHire — 03. Content-Based Job Recommender
In this notebook, we build, evaluate, and test the Core Recommender Engine:
- **Vector Space Model (TF-IDF)** on the preprocessed job corpus.
- **Cosine Similarity Matrix** ranking candidate resumes against thousands of job postings.
- **Skill Gap Analysis**: Deterministic matching of required vs candidate skills.
- **Deduplication & Top-N Filtering** (Top 5, 10, 15).
- **Explainability**: Transparent technical explanations for why jobs match.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.models.recommender import JobRecommender
from src.features.text_features import extract_skills

# Load cached recommender
recommender = JobRecommender.load()
print(f"Recommender loaded successfully. Indexed vacancies: {len(recommender.jobs_df)}")
print(f"TF-IDF matrix shape: {recommender.job_matrix.shape}")


## 1. Test Recommendations for a Data Science Profile


In [ ]:
ds_resume = '''
Alex Morgan - Data Scientist
Proficient in Python, SQL, Machine Learning, Deep Learning, Scikit-learn, Pandas, Tableau, Statistics, and Docker.
Experience in developing predictive churn models and NLP text classifiers.
'''

results_ds = recommender.recommend(ds_resume, top_n=5)
print(f"Generated {len(results_ds)} recommendations:\n")

for job in results_ds:
    print(f"#{job['rank']}: {job['title']} at {job['company']}")
    print(f"   Match %: {job['match_percentage']}% | Experience: {job['experience']}")
    print(f"   Matched Skills: {job['matched_skills']}")
    print(f"   Missing Skills: {job['missing_skills']}")
    print(f"   Explanation: {job['explanation']}\n")


## 2. Test Recommendations for a Full Stack Web Developer Profile


In [ ]:
web_resume = '''
Priya Sharma - Full Stack Software Engineer
Core Skills: Python, Django, React.js, JavaScript, TypeScript, Node.js, PostgreSQL, REST API, HTML, CSS, Git, Docker, CI/CD.
Experience building scalable responsive web applications and RESTful microservices.
'''

results_web = recommender.recommend(web_resume, top_n=5)
print(f"Generated {len(results_web)} recommendations for Web Dev Profile:\n")

for job in results_web:
    print(f"#{job['rank']}: {job['title']} at {job['company']}")
    print(f"   Match %: {job['match_percentage']}% | Coverage: {job['skill_coverage_pct']}%")
    print(f"   Matched Skills: {job['matched_skills']}")
    print(f"   Missing Skills: {job['missing_skills']}\n")


## 3. Qualitative Evaluation & Precision@K Check


In [ ]:
# Verify that top recommendations for Data Science resume contain data science/ML roles
relevant_keywords = ['data', 'scientist', 'machine learning', 'analytics', 'python']

hits = 0
for job in results_ds:
    title_lower = job['title'].lower()
    if any(kw in title_lower for kw in relevant_keywords):
        hits += 1

precision_at_5 = hits / len(results_ds)
print(f"Precision@5 on sample Data Science profile: {precision_at_5 * 100:.1f}% ({hits}/{len(results_ds)} relevant)")
